# SOB4ES - RegressorChain (Random Forest base)
### CRISP-ML(Q) - Fase 3 y 4: Desarrollo y evaluacion de los modelos

Este notebook implementa un **RegressorChain** usando Random Forest como modelo base.

La idea es encadenar las predicciones: el modelo primero predice el target 1, luego usa esa prediccion junto a las variables originales para predecir el target 2, y asi sucesivamente. Esto introduce correlacion real entre targets: cada prediccion tiene acceso a las predicciones anteriores en la cadena.

El problema del encadenamiento simple es que los targets al principio de la cadena tienen menos informacion que los del final. Para compensarlo se usa un **ensemble de cadenas** donde cada cadena usa un orden diferente y aleatorio de los targets. Al promediar los resultados, los errores acumulados por el orden se cancelan entre si.

**Datasets utilizados:**
- `train.csv` - Entrenamiento (70 %, estratificado por pais)
- `test.csv`  - Tuning / validacion cruzada (15 %)
- `eval.csv`  - Evaluacion final imparcial (15 %)

**Estructura del notebook:**
1. Configuracion
2. Carga de datos
3. Tuning del modelo base
4. Validacion cruzada
5. Analisis de la cadena: importancia de variables y orden
6. Entrenamiento de produccion (ensemble de cadenas)
7. Evaluacion final

---
## 1. Configuracion del entorno


In [1]:
import os
import time
import warnings
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import RegressorChain
from sklearn.model_selection import RandomizedSearchCV, cross_validate, RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')

DATA_DIR   = 'input/'
MODELS_DIR = 'output/models/chain/'
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

N_CHAINS = 10  # numero de cadenas en el ensemble; mas cadenas = mas estable pero mas lento

TARGETS = [
    # Shannon diversity index
    'nematode_shannon_z',
    'macro_shannon_z',
    'earthworm_shannon_z',
    'orib_shannon_z',
    'meso_shannon_z',
    'coll_shannon_z',
    'bac_shannon_z',
    'fun_shannon_z',
    'euk_shannon_z',
    'oomy_shannon_z',
    'cerc_shannon_z',
    # Richness
    'macro_order_richness_z',
    'earthworm_richness_z',
    'orib_species_richness_z',
    'meso_species_richness_z',
    'coll_species_richness_z',
    'bac_asv_richness_z',
    'fun_asv_richness_z',
    'euk_asv_richness_z',
    'oomy_asv_richness_z',
    'cerc_asv_richness_z'
]


FEATURES_AUTORIZADAS = [
    'total_plant_cover_z',
    'clay_content_z',
    'silt_content_z',
    'sand_content_z',
    'aggregate_stability_z',
    'bulk_density_z',
    'soil_moisture_z',
    'as_z',
    'cu_z',
    'k_z',
    'mo_z',
    'ni_z',
    'p_z',
    'pb_z',
    'zn_z',
    'soil_ph_z',
    'plot_total_organic_c_z',
    'plot_total_n_z',
    'gee_temp_media_C_z',
    'gee_humedad_rel_pct_z',
    'gee_ndvi_verano_z',
    'dem_elevacion_m_z',
    'dem_pendiente_deg_z',
    'dem_orientacion_deg_z',
    'eu_clay_content_z',
    'eu_sand_content_z',
    'eu_silt_content_z',
    'eu_water_holding_capacity_z',
    'eu_cn_ratio_z',
    'eu_p_z',
    'eu_ph_z',
    'eu_as_z',
    'eu_organic_carbon_octop_z',
    'eu_zn_z'
]


print('Librerias y variables cargadas.')


Librerias y variables cargadas.


## 2. Carga de datos

Se cargan los tres datasets pre-generados (division 70/15/15, estratificada por pais).

| Archivo    | Uso                                      | Filas aprox. |
|------------|------------------------------------------|--------------|
| `train.csv`| Ajuste del modelo                        | 299          |
| `test.csv` | Tuning / validacion cruzada              | 64           |
| `eval.csv` | Evaluacion final imparcial               | 65           |

El dataset `eval.csv` no se utiliza hasta la seccion 7.

In [2]:
def cargar_csv(ruta, nombre):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f'No se encontro: {ruta}')
    try:
        df = pd.read_csv(ruta, sep=',')
        if len(df.columns) < 5:
            df = pd.read_csv(ruta, sep=';')
    except Exception:
        df = pd.read_csv(ruta, sep=';')
    print(f'  {nombre}: {df.shape[0]} filas x {df.shape[1]} columnas')
    return df

print('Cargando datasets...')
df_train = cargar_csv(os.path.join(DATA_DIR, 'train.csv'), 'train.csv')
df_test  = cargar_csv(os.path.join(DATA_DIR, 'test.csv'),  'test.csv')
df_eval  = cargar_csv(os.path.join(DATA_DIR, 'eval.csv'),  'eval.csv')

X_train = df_train[FEATURES_AUTORIZADAS]
X_test  = df_test[FEATURES_AUTORIZADAS]
X_eval  = df_eval[FEATURES_AUTORIZADAS]

# y es una matriz 2D (n_muestras x n_targets) para todos los modelos de esta fase
y_train = df_train[TARGETS]
y_test  = df_test[TARGETS]
y_eval  = df_eval[TARGETS]

print(f'\nDatasets cargados.')
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_eval:  {X_eval.shape}   |  y_eval:  {y_eval.shape}')


Cargando datasets...
  train.csv: 299 filas x 71 columnas
  test.csv: 64 filas x 71 columnas
  eval.csv: 65 filas x 71 columnas

Datasets cargados.
X_train: (299, 34)  |  y_train: (299, 11)
X_eval:  (65, 34)   |  y_eval:  (65, 11)


## 3. Tuning del modelo base

`RegressorChain` encapsula un modelo base que se replica una vez por target. El tuning se hace sobre ese modelo base (Random Forest individual) usando un unico target de referencia. Los parametros optimos se usaran para construir cada cadena del ensemble.

Nota: el tuning del modelo base es suficiente porque `RegressorChain` no tiene hiperparametros propios mas alla del orden de la cadena, que se aleatoriza en produccion.

In [3]:
print('Iniciando busqueda de hiperparametros del modelo base...')
print('Este proceso puede tardar varios minutos.\n')

param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
    'bootstrap':         [True, False]
}

y_ref = y_train['earthworm_shannon_z'].values

buscador = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=50, scoring='r2', cv=5,
    verbose=1, random_state=42, n_jobs=-1
)

start = time.time()
buscador.fit(X_train, y_ref)

print(f'\nBusqueda completada en {(time.time()-start)/60:.1f} minutos.')
print(f'Mejor R2 de referencia: {buscador.best_score_:.4f}')
print(f'Hiperparametros optimos: {buscador.best_params_}')

BASE_PARAMS = buscador.best_params_
BASE_PARAMS['n_jobs'] = -1


Iniciando busqueda de hiperparametros del modelo base...
Este proceso puede tardar varios minutos.

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/home/aka/Desktop/GitHub/TFG-SOB4ES/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/aka/Desktop/GitHub/TFG-SOB4ES/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/aka/Desktop/GitHub/TFG-SOB4ES/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/aka/Deskto


Busqueda completada en 1.5 minutos.
Mejor R2 de referencia: 0.5056
Hiperparametros optimos: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': None, 'bootstrap': True}


## 4. Validacion cruzada

Se evalua la robustez de una cadena unica (orden por defecto) con validacion cruzada repetida (5 pliegues x 3 repeticiones) sobre `X_train`. El ensemble de cadenas aleatorias que se entrena en produccion sera mas robusto que esta cadena unica, por lo que estos resultados son una estimacion conservadora del rendimiento final.

In [4]:
print('Validacion cruzada (cadena unica, orden por defecto)...\n')

# Cadena unica para validacion: order=None usa el orden original de TARGETS
cadena_cv   = RegressorChain(RandomForestRegressor(**BASE_PARAMS), order=None)
cv_splitter = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

resultados = cross_validate(
    estimator=cadena_cv,
    X=X_train, y=y_train,
    cv=cv_splitter,
    scoring='r2',
    return_train_score=True
)

r2_train = np.mean(resultados['train_score'])
r2_cv    = np.mean(resultados['test_score'])
r2_std   = np.std(resultados['test_score'])

print(f'  Train R2 (promedio targets): {r2_train:.4f}')
print(f'  Val   R2 (promedio targets): {r2_cv:.4f} +/- {r2_std:.4f}')

# R2 individual por target
cadena_cv.fit(X_train, y_train)
y_pred_train = cadena_cv.predict(X_train)
print('\n  R2 por target (train, orden por defecto):')
for i, t in enumerate(TARGETS):
    r2 = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
    print(f'    {t:30} {r2:.4f}')


Validacion cruzada (cadena unica, orden por defecto)...

  Train R2 (promedio targets): 0.6489
  Val   R2 (promedio targets): 0.1560 +/- 0.0263

  R2 por target (train, orden por defecto):
    nematode_shannon_z             0.7471
    macro_shannon_z                0.7459
    earthworm_shannon_z            0.8267
    orib_shannon_z                 0.6711
    meso_shannon_z                 0.5404
    coll_shannon_z                 0.6773
    bac_shannon_z                  0.5322
    fun_shannon_z                  0.6177
    euk_shannon_z                  0.6665
    oomy_shannon_z                 0.6788
    cerc_shannon_z                 0.4826


## 5. Analisis de la cadena

Se analiza el efecto del orden de la cadena en el rendimiento por target. Los targets al principio de la cadena solo tienen acceso a las variables originales, mientras que los del final tienen ademas las predicciones de todos los anteriores.

Se muestra tambien la importancia de variables del modelo base para el primer target de la cadena, donde no hay predicciones previas disponibles y la dificultad es maxima.

In [5]:
print('Analisis del efecto del orden en la cadena...\n')

# Comparamos el R2 de cada target segun su posicion en la cadena (orden por defecto)
print('  Posicion en cadena -> target -> R2 (train):')
for i, t in enumerate(TARGETS):
    r2 = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
    print(f'    Posicion {i+1:2d} | {t:30} R2: {r2:.4f}')

# Importancia de variables del modelo base para el primer target
# (el mas dificil: solo tiene las variables originales, sin predicciones previas)
primer_modelo = cadena_cv.estimators_[0]
imp = pd.Series(primer_modelo.feature_importances_, index=FEATURES_AUTORIZADAS)

print('\n  Top 10 variables para el primer target de la cadena:')
print(imp.nlargest(10).round(4).to_string())


Analisis del efecto del orden en la cadena...

  Posicion en cadena -> target -> R2 (train):
    Posicion  1 | nematode_shannon_z             R2: 0.7471
    Posicion  2 | macro_shannon_z                R2: 0.7459
    Posicion  3 | earthworm_shannon_z            R2: 0.8267
    Posicion  4 | orib_shannon_z                 R2: 0.6711
    Posicion  5 | meso_shannon_z                 R2: 0.5404
    Posicion  6 | coll_shannon_z                 R2: 0.6773
    Posicion  7 | bac_shannon_z                  R2: 0.5322
    Posicion  8 | fun_shannon_z                  R2: 0.6177
    Posicion  9 | euk_shannon_z                  R2: 0.6665
    Posicion 10 | oomy_shannon_z                 R2: 0.6788
    Posicion 11 | cerc_shannon_z                 R2: 0.4826

  Top 10 variables para el primer target de la cadena:
sand_content_z           0.1545
silt_content_z           0.1039
soil_ph_z                0.0677
gee_humedad_rel_pct_z    0.0594
aggregate_stability_z    0.0544
gee_temp_media_C_z       0.0472

## 6. Entrenamiento de produccion (ensemble de cadenas)

Se entrenan 10 cadenas con ordenes aleatorios distintos. Cada cadena usa el mismo modelo base (Random Forest con los hiperparametros optimos) pero un orden diferente de los targets. Al promediar las predicciones de todas las cadenas, se compensan los errores de acumulacion que afectan a los targets al principio de cada cadena.

In [6]:
print(f'Entrenamiento de produccion ({N_CHAINS} cadenas con ordenes aleatorios)...')
print('-' * 70)

global_start = time.time()
ordenes_usados = []

for chain_id in range(N_CHAINS):
    t0 = time.time()

    # Cada cadena usa un orden aleatorio distinto de los 11 targets
    orden = list(np.random.RandomState(42 + chain_id).permutation(len(TARGETS)))
    ordenes_usados.append(orden)

    base_model = RandomForestRegressor(**{**BASE_PARAMS, 'random_state': 42 + chain_id})
    cadena     = RegressorChain(base_model, order=orden)
    cadena.fit(X_train, y_train)

    joblib.dump(cadena, os.path.join(MODELS_DIR, f'chain_{chain_id}.pkl'))
    print(f'  Cadena {chain_id+1:2d}/{N_CHAINS} | orden: {orden} | ({time.time()-t0:.1f}s)')

# Guardamos los ordenes para referencia futura
joblib.dump(ordenes_usados, os.path.join(MODELS_DIR, 'ordenes.pkl'))

print('-' * 70)
print(f'Produccion completada en {(time.time()-global_start)/60:.1f} minutos.')


Entrenamiento de produccion (10 cadenas con ordenes aleatorios)...
----------------------------------------------------------------------
  Cadena  1/10 | orden: [np.int64(5), np.int64(0), np.int64(9), np.int64(10), np.int64(2), np.int64(1), np.int64(8), np.int64(4), np.int64(7), np.int64(3), np.int64(6)] | (4.2s)
  Cadena  2/10 | orden: [np.int64(10), np.int64(7), np.int64(6), np.int64(8), np.int64(3), np.int64(9), np.int64(2), np.int64(5), np.int64(1), np.int64(0), np.int64(4)] | (4.5s)
  Cadena  3/10 | orden: [np.int64(8), np.int64(7), np.int64(2), np.int64(10), np.int64(0), np.int64(6), np.int64(9), np.int64(5), np.int64(1), np.int64(3), np.int64(4)] | (4.2s)
  Cadena  4/10 | orden: [np.int64(2), np.int64(6), np.int64(8), np.int64(7), np.int64(9), np.int64(1), np.int64(4), np.int64(10), np.int64(5), np.int64(0), np.int64(3)] | (3.9s)
  Cadena  5/10 | orden: [np.int64(10), np.int64(9), np.int64(0), np.int64(1), np.int64(6), np.int64(7), np.int64(3), np.int64(2), np.int64(4), np.int6

## 7. Evaluacion final y prueba de funcionamiento

Esta seccion tiene dos partes:

**Parte A - Evaluacion final**
Se mide por primera vez el rendimiento sobre `eval.csv`. Se muestran metricas globales (promedio sobre los 11 targets) y R2 individual por target.

**Parte B - Smoke test**
Se verificia que el pipeline funciona de extremo a extremo con 3 filas reales de `eval.csv`.

---
**Como leer el indice Shannon H':**
Cada target es el indice de diversidad de Shannon de un grupo biologico distinto. El valor va de `0.0` (sin diversidad) a `4.0-5.0` (alta diversidad). Un valor tipico en suelos agricolas europeos se situa entre `1.5` y `2.5`.

In [7]:
# ─────────────────────────────────────────────────────────────────────────
# PARTE A: Evaluacion final
# Cargamos las N cadenas y promediamos sus predicciones sobre eval.csv.
# ─────────────────────────────────────────────────────────────────────────
print('Evaluacion final sobre eval.csv')
print('-' * 60)

preds_eval = []
for chain_id in range(N_CHAINS):
    ruta   = os.path.join(MODELS_DIR, f'chain_{chain_id}.pkl')
    cadena = joblib.load(ruta)
    preds_eval.append(cadena.predict(X_eval))

# Promediamos sobre todas las cadenas: shape (N_CHAINS, n_eval, 11) -> (n_eval, 11)
y_pred_eval = np.mean(preds_eval, axis=0)
y_true_eval = y_eval.values

r2_global   = r2_score(y_true_eval, y_pred_eval, multioutput='uniform_average')
rmse_global = np.sqrt(mean_squared_error(y_true_eval, y_pred_eval, multioutput='uniform_average'))
mae_global  = mean_absolute_error(y_true_eval, y_pred_eval, multioutput='uniform_average')

print(f'  R2   global: {r2_global:.4f}')
print(f'  RMSE global: {rmse_global:.4f} puntos de Shannon H\'')
print(f'  MAE  global: {mae_global:.4f} puntos de Shannon H\'')

print('\n  R2 por target:')
r2_por_target = r2_score(y_true_eval, y_pred_eval, multioutput='raw_values')
for t, r2 in zip(TARGETS, r2_por_target):
    print(f'    {t:30} {r2:.4f}')

# ─────────────────────────────────────────────────────────────────────────
# PARTE B: Smoke test con datos reales
# ─────────────────────────────────────────────────────────────────────────
print('\nSmoke test - Inferencia con datos reales')
print('-' * 60)

try:
    df_new_data      = X_eval.sample(n=3, random_state=42)
    indices_elegidos = df_new_data.index
    valores_reales   = y_eval.loc[indices_elegidos].values

    votos = []
    for chain_id in range(N_CHAINS):
        ruta   = os.path.join(MODELS_DIR, f'chain_{chain_id}.pkl')
        cadena = joblib.load(ruta)
        pred   = cadena.predict(df_new_data)
        votos.append(pred)
        # pred es una matriz (3 muestras x 11 targets)
        print(f'  Cadena {chain_id+1:2d} | '
              f'Fila {indices_elegidos[0]}: {pred[0, 2]:.4f} | '
              f'Fila {indices_elegidos[1]}: {pred[1, 2]:.4f} | '
              f'Fila {indices_elegidos[2]}: {pred[2, 2]:.4f}  (earthworm_shannon_z)')

    consenso = np.mean(votos, axis=0)
    print('-' * 60)
    print('  Resultados detallados por muestra (earthworm_shannon_z):')
    print('-' * 60)

    idx_earth = TARGETS.index('earthworm_shannon_z')
    for i, idx in enumerate(indices_elegidos):
        pred_val = consenso[i, idx_earth]
        real_val = y_eval.loc[idx, 'earthworm_shannon_z']
        error    = abs(pred_val - real_val)
        print(f'    Fila {idx:<4} | Prediccion: {pred_val:.4f} | '
              f'Valor Real: {real_val:.4f} | Error absoluto: {error:.4f}')

    print('-' * 60)
    print('\nSmoke test superado. El ensemble de cadenas funciona correctamente.')

except Exception as e:
    print(f'Error en el smoke test: {str(e)}')


Evaluacion final sobre eval.csv
------------------------------------------------------------
  R2   global: 0.0843
  RMSE global: 0.9331 puntos de Shannon H'
  MAE  global: 0.6972 puntos de Shannon H'

  R2 por target:
    nematode_shannon_z             0.1461
    macro_shannon_z                0.3474
    earthworm_shannon_z            0.5345
    orib_shannon_z                 0.2207
    meso_shannon_z                 -0.1497
    coll_shannon_z                 -0.3895
    bac_shannon_z                  -0.0187
    fun_shannon_z                  -0.0334
    euk_shannon_z                  0.1746
    oomy_shannon_z                 0.0668
    cerc_shannon_z                 0.0281

Smoke test - Inferencia con datos reales
------------------------------------------------------------
  Cadena  1 | Fila 53: 0.6150 | Fila 60: 0.9805 | Fila 0: -0.5041  (earthworm_shannon_z)
  Cadena  2 | Fila 53: 0.6046 | Fila 60: 0.9688 | Fila 0: -0.5197  (earthworm_shannon_z)
  Cadena  3 | Fila 53: 0.5882 | Fi